In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.covariance import LedoitWolf
import matplotlib.pyplot as plt

# ==============================
# CONFIG
# ==============================
CAPITAL = 50_000_000
RIDGE_LAMBDA = 1.0
TRAIN_WINDOW = 300
ALPHA_WINDOW = 60
VAM_WINDOW = 20
STRATEGY_CAP = 0.6

# ==============================
# DATA (replace with real prices)
# ==============================
np.random.seed(42)
dates = pd.date_range("2015-01-01", periods=2500)
tickers = ["A", "B", "C"]

prices = pd.DataFrame(
    100 * np.cumprod(1 + 0.001 * np.random.randn(len(dates), len(tickers)), axis=0),
    index=dates,
    columns=tickers
)

returns = prices.pct_change().dropna()

# ==============================
# STRATEGY SIGNALS
# ==============================
def ema_signal(close, span=20):
    ema = close.ewm(span=span).mean()
    return np.sign(close - ema)

def momentum_signal(close, lookback=10):
    return np.sign(close.pct_change(lookback))

def buy_hold_signal(close):
    return np.ones(len(close))

STRATEGIES = {
    "EMA": ema_signal,
    "MOM": momentum_signal,
    "BH": buy_hold_signal
}

# ==============================
# CAP HANDLER (PROPER)
# ==============================
def apply_caps(weights, cap):
    w = weights.copy()
    while True:
        capped = w > cap
        if not capped.any():
            break
        excess = w[capped].sum() - cap * capped.sum()
        w[capped] = cap
        uncapped = ~capped
        w[uncapped] += excess * w[uncapped] / w[uncapped].sum()
    return w

# ==============================
# BACKTEST
# ==============================
portfolio_equity = [CAPITAL]
allocations_log = []

for t in range(TRAIN_WINDOW + ALPHA_WINDOW, len(returns)):

    # -------- STOCK-LEVEL BUDGETS --------
    lw = LedoitWolf().fit(returns.iloc[t-TRAIN_WINDOW:t])
    cov = lw.covariance_
    mu = returns.iloc[t-TRAIN_WINDOW:t].mean().values
    w_stock = np.maximum(np.linalg.inv(cov) @ mu, 0)
    w_stock /= w_stock.sum()
    stock_budgets = dict(zip(tickers, CAPITAL * w_stock))

    day_allocations = {}

    # -------- STRATEGY LEVEL --------
    for X in tickers:
        close = prices[X].iloc[:t]
        r = returns[X].iloc[:t]

        scores = []

        for name, strat in STRATEGIES.items():
            signal = strat(close).shift(1)
            strat_ret = signal * r

            excess = strat_ret - r

            if len(strat_ret.dropna()) < TRAIN_WINDOW:
                scores.append(0)
                continue

            alpha = (
                excess.iloc[-ALPHA_WINDOW:].mean()
                / excess.iloc[-ALPHA_WINDOW:].std()
            )

            vam = (
                strat_ret.iloc[-VAM_WINDOW:].mean()
                / strat_ret.iloc[-VAM_WINDOW:].std()
            )

            # ML TRAINING
            X_train = pd.DataFrame({
                "alpha": excess.rolling(ALPHA_WINDOW).mean(),
                "vam": strat_ret.rolling(VAM_WINDOW).mean()
            }).dropna()

            y_train = strat_ret.shift(-1).loc[X_train.index]

            model = Ridge(alpha=RIDGE_LAMBDA)
            model.fit(X_train.iloc[-TRAIN_WINDOW:], y_train.iloc[-TRAIN_WINDOW:])

            pred = model.predict([[alpha, vam]])[0]
            sigma = np.std(y_train.iloc[-TRAIN_WINDOW:] - model.predict(X_train.iloc[-TRAIN_WINDOW:]))

            score = max(pred / sigma, 0)
            scores.append(score)

        scores = np.array(scores)
        if scores.sum() == 0:
            weights = np.zeros_like(scores)
        else:
            weights = scores / scores.sum()

        weights = apply_caps(weights, STRATEGY_CAP)

        for i, strat_name in enumerate(STRATEGIES.keys()):
            day_allocations[(X, strat_name)] = weights[i] * stock_budgets[X]

    # -------- DAILY PNL --------
    pnl = 0
    for (X, strat_name), alloc in day_allocations.items():
        sig = STRATEGIES[strat_name](prices[X].iloc[:t]).iloc[-1]
        pnl += alloc * sig * returns[X].iloc[t]

    portfolio_equity.append(portfolio_equity[-1] + pnl)
    allocations_log.append(day_allocations)

# ==============================
# OUTPUT
# ==============================
equity = pd.Series(portfolio_equity, index=prices.index[-len(portfolio_equity):])

equity.plot(title="Portfolio Equity Curve")
plt.show()

print("Final Equity:", equity.iloc[-1])


ValueError: Input y contains NaN.